# `c07_gr200` — Graduation Rates at 200 Percent of Normal Time

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `GR200_23` |
| Reference period | Status as of August 31, 2023 for the 2015 entering cohort at 4-year institutions and the 2019 entering cohort at less-than-4-year institutions |
| Curated grain | `UNITID` |
| Output | `data/curated/c07_gr200.parquet` |

The extended window answers a different question from the 150 percent rate: how many students finish eventually. The gap between BAGR150 and BAGR200 is itself the interesting feature, because it separates institutions serving students who complete slowly from those whose students do not complete at all.

> **Pitfall.** These are already percentages, not counts, so they cannot be aggregated across institutions by averaging without weighting by cohort size. The monotonicity check is the cheapest available test that a join has not gone wrong.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c07_gr200"
TABLES = ['GR200_23']
GRAIN = ['UNITID']
REFERENCE_PERIOD = 'Status as of August 31, 2023 for the 2015 entering cohort at 4-year institutions and the 2019 entering cohort at less-than-4-year institutions'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,GR200_23,101013,56c2eee675f7201e3c87314ad1bb5b20348cb766e35fa1...,2026-09-24T17:19:26+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'cohort year 2015',
    table=TABLES[0],
)
print(intro[:600])

File documentation for graduation rate data, 200% of normal time to complete - cohort year 2015 (4-year institutions) - cohort year 2019 (less than 4-year institutions) : 2023
(Provisional release)
Filename GR200_23
Overview This file contains the graduation rate status as of August 31, 2023 for the cohort of full-time, first-time degree/certificate-seeking undergraduates. Data for four year institutions include the number of bachelor degree-seeking students who were enrolled in 2015, the number of bachelor degree seeking students who completed a bachelor's degree within 100, 150  or 200 perce


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

27 variables documented, 0 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,BAREVCT,"Revised bachelor's degree-seeking cohort, (coh..."
2,BAEXCLU,Exclusions from bachelor's degree-seeking coho...
3,BAAC150,Adjusted bachelor's degree-seeking cohort with...
4,BANC100,Number completed a bachelor's degree within 10...
5,BAGR100,4-year Graduation rate - bachelor's degree wit...
6,BANC150,Number completed a bachelor's degree within 15...
7,BAGR150,6-year Graduation rate - bachelor's degree wit...
8,BAAEXCL,Additional exclusions from bachelor's degree-s...
9,BAAC200,Adjusted bachelor's degree-seeking cohort with...


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'BAGR100', 'BAGR150', 'BAGR200', 'L4GR100', 'L4GR150', 'L4GR200']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (4954, 53)
schema: unchanged | added [] | removed []


,UNITID,BAGR100,BAGR150,BAGR200,L4GR100,L4GR150,L4GR200
0,100654,11.0,28.0,30.0,NaN,NaN,NaN
1,100663,42.0,62.0,65.0,NaN,NaN,NaN
2,100690,67.0,67.0,67.0,NaN,NaN,NaN
3,100706,35.0,61.0,63.0,NaN,NaN,NaN
4,100724,10.0,28.0,30.0,NaN,NaN,NaN


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in []:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 6 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
0,XBAGR100,A,2988,0.6031
1,XBAGR100,R,1965,0.3966
2,XBAGR100,Z,1,0.0002
3,XBAGR150,A,2988,0.6031
4,XBAGR150,R,1965,0.3966
5,XBAGR150,Z,1,0.0002
6,XBAGR200,A,2988,0.6031
7,XBAGR200,R,1965,0.3966
8,XBAGR200,Z,1,0.0002
9,XL4GR100,R,2986,0.6027


Columns under 90% reported — interpret with care:


column
XBAGR100    0.3966
XBAGR150    0.3966
XBAGR200    0.3966
XL4GR100    0.6027
XL4GR150    0.6027
XL4GR200    0.6027
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = []

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,UNITID,BAGR100,BAGR150,BAGR200,L4GR100,L4GR150,L4GR200
0,100654,11.0,28.0,30.0,NaN,NaN,NaN
1,100663,42.0,62.0,65.0,NaN,NaN,NaN
2,100690,67.0,67.0,67.0,NaN,NaN,NaN
3,100706,35.0,61.0,63.0,NaN,NaN,NaN
4,100724,10.0,28.0,30.0,NaN,NaN,NaN


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID'] -> 4,954 rows, 0 duplicated


,UNITID,BAGR100,BAGR150,BAGR200,L4GR100,L4GR150,L4GR200
0,100654,11.0,28.0,30.0,NaN,NaN,NaN
1,100663,42.0,62.0,65.0,NaN,NaN,NaN
2,100690,67.0,67.0,67.0,NaN,NaN,NaN
3,100706,35.0,61.0,63.0,NaN,NaN,NaN
4,100724,10.0,28.0,30.0,NaN,NaN,NaN


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.in_range('BAGR200', 0, 100, severity='warn'),
    iu.Rule('monotone_200_ge_150', lambda d: pd.to_numeric(d.BAGR200, errors='coerce') < pd.to_numeric(d.BAGR150, errors='coerce'), note='200% rate cannot fall below the 150% rate'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.0,Declared grain must be unique
1,"in_range(BAGR200,0,100)",pass,0,0.0,Value plausibility bound
2,monotone_200_ge_150,pass,0,0.0,200% rate cannot fall below the 150% rate


PASSED


Report(table='c07_gr200', rows=4954, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(BAGR200,0,100)', 'severity': 'warn', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'monotone_200_ge_150', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': '200% rate cannot fall below the 150% rate', 'status': 'pass'}], generated_utc='2026-09-24T17:19:26+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='These are already percentages, not counts, so they cannot be aggregated across institutions by averaging without weighting by cohort size. The monotonicity check is the cheapest available test that a join has not gone wrong.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c07_gr200.parquet (4,954 rows x 7 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. These are already percentages, not counts, so they cannot be aggregated across institutions by averaging without weighting by cohort size. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.